# Task 3: Partition Pruning Experiment (Queries 1, 6)

- February 2024 queries are run on a month-partitioned Gold table and an identical unpartitioned Delta copy.

## Connect to Spark

In [18]:
import sys

from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.benchmarking import measure_registered_query, rows_match
from analytics.runner import (
    DEFAULT_INTEGRATED_TABLE_PATH,
    INTEGRATED_TABLE_VIEW,
    execute_registered_query,
    register_integrated_taxi_trips,
)

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('Assignment2-PartitionPruning-Query1-6')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark

## Define the two layouts

- February is used because the data covers January-March 2024. 
- Partition prunnig by skipping January and March.

In [ ]:
PARTITIONED_TABLE_PATH = DEFAULT_INTEGRATED_TABLE_PATH
UNPARTITIONED_TABLE_PATH = (
    's3a://delta-data/gold/gold_integrated_taxi_trips_unpartitioned'
)
QUERY_NAMES = (
    'query_01_monthly_zone_demand_february_2024',
    'query_06_monthly_taxi_demand_trend_february_2024',
)
REPETITIONS = 9

## Create the unpartitioned layout

In [20]:
partitioned_table = spark.read.format('delta').load(PARTITIONED_TABLE_PATH)

(
    partitioned_table.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(UNPARTITIONED_TABLE_PATH)
)

unpartitioned_table = (
    spark.read.format('delta').load(UNPARTITIONED_TABLE_PATH)
)

assert partitioned_table.count() == unpartitioned_table.count()
print('Unpartitioned layout created with the same number of rows.')

Unpartitioned layout created with the same number of rows.


## Measure both layouts

Each layout receives one warm-up followed by three timed runs.

In [ ]:
def measure_layout(table_path):
    spark.catalog.clearCache()
    register_integrated_taxi_trips(spark, table_path)
    return {
        query_name: measure_registered_query(spark, query_name, REPETITIONS)
        for query_name in QUERY_NAMES
    }

partitioned_results = measure_layout(PARTITIONED_TABLE_PATH)
unpartitioned_results = measure_layout(UNPARTITIONED_TABLE_PATH)

for query_name in QUERY_NAMES:
    assert rows_match(
        partitioned_results[query_name]['rows'],
        unpartitioned_results[query_name]['rows'],
    ), f'Partitioned and unpartitioned results differ for {query_name}.'

print('Both layouts return the same results within float tolerance.')

Both layouts return the same results within float tolerance.


## Partitioned physical plans

In [22]:
register_integrated_taxi_trips(spark, PARTITIONED_TABLE_PATH)
for query_name in QUERY_NAMES:
    print(f'\n--- {query_name} ---')
    execute_registered_query(spark, query_name).explain('formatted')


--- query_01_monthly_zone_demand_february ---
== Physical Plan ==
AdaptiveSparkPlan (7)
+- Sort (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Exchange (3)
            +- HashAggregate (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [pickup_zone_name#16351, pickup_month#16345]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips]
PartitionFilters: [isnotnull(pickup_month#16345), (cast(pickup_month#16345 as bigint) = 2)]
ReadSchema: struct<pickup_zone_name:string>

(2) HashAggregate
Input [2]: [pickup_zone_name#16351, pickup_month#16345]
Keys [2]: [pickup_zone_name#16351, pickup_month#16345]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#16518L]
Results [3]: [pickup_zone_name#16351, pickup_month#16345, count#16519L]

(3) Exchange
Input [3]: [pickup_zone_name#16351, pickup_month#16345, count#16519L]
Arguments: hashpartitioning(pickup_zone_name#16351, pickup_month#16345, 4), ENSURE_REQU

## Unpartitioned physical plans

- It does not contain a `PartitionFilters` entry.

In [23]:
register_integrated_taxi_trips(spark, UNPARTITIONED_TABLE_PATH)
for query_name in QUERY_NAMES:
    print(f'\n--- {query_name} ---')
    execute_registered_query(spark, query_name).explain('formatted')


--- query_01_monthly_zone_demand_february ---
== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Filter (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [pickup_month#16673, pickup_zone_name#16679]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips_unpartitioned]
PushedFilters: [IsNotNull(pickup_month)]
ReadSchema: struct<pickup_month:string,pickup_zone_name:string>

(2) Filter
Input [2]: [pickup_month#16673, pickup_zone_name#16679]
Condition : (isnotnull(pickup_month#16673) AND (cast(pickup_month#16673 as bigint) = 2))

(3) HashAggregate
Input [2]: [pickup_month#16673, pickup_zone_name#16679]
Keys [2]: [pickup_zone_name#16679, pickup_month#16673]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#16842L]
Results [3]: [pickup_zone_name#16679, pickup_month#16673, count#16843L]

## Benchmark summary

In [24]:
summary = []
for query_name in QUERY_NAMES:
    partitioned = partitioned_results[query_name]
    unpartitioned = unpartitioned_results[query_name]
    summary.append({
        'query': query_name,
        'partitioned_samples_seconds': [
            round(value, 3) for value in partitioned['samples_seconds']
        ],
        'partitioned_median_seconds': round(partitioned['median_seconds'], 3),
        'unpartitioned_samples_seconds': [
            round(value, 3) for value in unpartitioned['samples_seconds']
        ],
        'unpartitioned_median_seconds': round(unpartitioned['median_seconds'], 3),
        'partitioned_speedup_factor': round(
            unpartitioned['median_seconds'] / partitioned['median_seconds'],
            2,
        ),
    })
summary

[{'query': 'query_01_monthly_zone_demand_february',
  'partitioned_samples_seconds': [1.223,
   1.171,
   1.034,
   0.907,
   1.013,
   0.827,
   0.986,
   0.851,
   0.76,
   0.924],
  'partitioned_median_seconds': 0.955,
  'unpartitioned_samples_seconds': [1.659,
   1.332,
   1.131,
   1.126,
   1.183,
   1.105,
   1.254,
   1.045,
   1.248,
   0.958],
  'unpartitioned_median_seconds': 1.157,
  'partitioned_speedup_factor': 1.21},
 {'query': 'query_06_monthly_taxi_demand_trend_february',
  'partitioned_samples_seconds': [0.714,
   0.784,
   0.556,
   0.686,
   0.559,
   0.6,
   0.66,
   0.545,
   0.732,
   0.73],
  'partitioned_median_seconds': 0.673,
  'unpartitioned_samples_seconds': [0.987,
   0.785,
   0.777,
   0.79,
   0.937,
   0.954,
   0.742,
   0.757,
   0.909,
   0.703],
  'unpartitioned_median_seconds': 0.788,
  'partitioned_speedup_factor': 1.17}]

## Cleanup

In [25]:
spark.catalog.clearCache()
spark.stop()